# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described by an [MLCommons Croissant](https://mlcommons.org/croissant/) schema, provided via the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed in the current environment
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and prepare for inspection using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let’s review information about the available record sets (`@id`), and which fields and columns are included.

In Croissant datasets, **record sets** represent the structured tabular data sources. Each record set has an `@id` that should be referenced for extraction and downstream operations. Fields represent named columns and use their own `@id`.

In [ ]:
# List all available record sets with their @id
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')}")

# Preview fields in each record set
print("\nFields in each record set (with @id):")
for rs in dataset.record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    for field in rs.get('field', []):
        # Each field is typically a dict with @id, name, and other metadata
        print(f"  - field @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")

## 3. Data Extraction
Let’s load data from the main record set(s) into a DataFrame for analysis. 

For this dataset, the primary record set is most likely a tabular CSV file describing the individual survey responses. We use its `@id` to extract the data.

**Note:** The identifiers and field names are printed in the previous block. Adjust the `record_sets` list below with all record set `@id`s you want to load.

In [ ]:
# Collect the record set @ids (from the overview above)
# For this dataset, there is likely one major record set. Grab all for demonstration.
record_sets = [rs['@id'] for rs in dataset.record_sets]

# For most survey datasets, it is typical there is one tabular record set
print(f"Record sets to be loaded: {record_sets}")

# Load the data into pandas DataFrames, referencing record set by @id
dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading records from record set @id: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records. Sample columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print("No records found.")

## 4. Exploratory Data Analysis (EDA)
Now we’ll perform some basic exploratory analysis and simple data processing.

### Choose a numeric field and a grouping (categorical) field

You can find the actual field names and their `@id` in the overview above, but common fields for a mental health survey dataset may include assessment scores, e.g., `gad_7_score`, `phq_9_score`, or demographic information such as `gender`, `village`, or `age`.

Replace the field IDs below if your dataset uses different ones.

In [ ]:
# Choose the main record set to analyze
# You can adapt 'main_record_set_id' based on the actual record sets printed above.
if record_sets:
    main_record_set_id = record_sets[0]
    df = dataframes[main_record_set_id]
    print(f"Using record set: {main_record_set_id}")

    # Select one numeric field (example: 'gad_7_score' or its field @id)
    possible_numeric_fields = [c for c in df.columns if any(x in c.lower() for x in ['score','age','phq','gad','psq'])]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
    else:
        print("Could not determine numeric field automatically, please set manually.")
        numeric_field = df.columns[0]  # fallback
    print(f"Numeric field selected: {numeric_field}")

    # Example filtering: select only records with score > 10
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score normalization):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if present (example: 'gender', 'village', etc.)
    possible_group_fields = [c for c in df.columns if any(x in c.lower() for x in ['gender','village','education','marital','category'])]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} by {group_field}:")
        display(grouped_df)
    else:
        print("No suitable group field found. Skipping groupby example.")

## 5. Visualization
Let’s plot the distribution of a selected numeric field and (optionally) see its distribution across a demographic group, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    df = dataframes[main_record_set_id]

    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], bins=20, kde=True, color='orchid')
    plt.xlabel(numeric_field)
    plt.title(f'Histogram of {numeric_field}')
    plt.show()

    # If a grouping field exists, show boxplot by group
    if possible_group_fields:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=df, x=possible_group_fields[0], y=numeric_field)
        plt.title(f"{numeric_field} by {possible_group_fields[0]}")
        plt.show()

## 6. Conclusion
- We loaded and inspected the FAIR² Mental Health Survey Dataset using the `mlcroissant` library, referencing all entities using their `@id` where relevant.
- The data provides a rich set of tabular survey results with demographic and mental health assessment information.
- We demonstrated basic extraction, filtering, normalization, analysis by demographic group, and visualization.

Further analyses can be conducted to answer research questions about mental health patterns, demographic disparities, and more, all with traceable links to the Croissant schema for reproducibility and FAIRness.